In [ ]:
"""
====================================================================================
Blockchain Transaction Preprocessing Pipeline
====================================================================================

Author: Farah Bassoumi
Version: 3.0
Target Environment: Ethereum Transaction Data (DeFi, MEV, Sybil detection, etc.)

------------------------------------------------------------------------------------
Objective
------------------------------------------------------------------------------------
This module implements a modular, auditable, and memory-efficient preprocessing pipeline
for Ethereum transaction datasets. It is designed to clean, optimize, and enrich 
transaction data before further analysis such as Sybil address classification, MEV 
pattern detection (e.g., sandwich attacks), or DeFi protocol monitoring.

------------------------------------------------------------------------------------
Key Features
------------------------------------------------------------------------------------
✔ Modular Design:
   - Each transformation step is encapsulated in its own function.
   - Enables unit testing, debugging, and partial reuse of the pipeline.

✔ Memory and Row Tracking:
   - At each step, logs reduction in dataset size and memory footprint.
   - Facilitates monitoring and optimization on large-scale datasets.

✔ Built-in Sybil Address Labeling:
   - Automatically merges external Sybil address labels for supervised learning tasks.

✔ Inclusion Time & Miner Detection:
   - Derives the exact on-chain inclusion time and flags miner-injected transactions
     (i.e., with `timepending = 0`), critical for MEV research.

------------------------------------------------------------------------------------
Pipeline Steps
------------------------------------------------------------------------------------
1. **Column Cleaning**
   - Removes malformed column names (e.g., containing `\n`, `,`).
   - Strips whitespace and removes duplicate column names.

2. **Column Reduction**
   - Drops non-essential columns (defined in `IRRELEVANT_COLUMNS`).

3. **Critical Row Filtering**
   - Ensures all critical fields are present before proceeding.

4. **Confirmed Transaction Filtering**
   - Keeps only confirmed transactions with valid timing metadata.

5. **Deduplication**
   - Removes duplicate rows and redundant regions.

6. **Sybil Labeling**
   - Joins with a labeled dataset of known Sybil addresses.

7. **Inclusion Time Computation**
   - Computes the actual time a transaction was confirmed on-chain.
   - Flags whether a transaction was injected by a miner.

8. **Final Cleanup**
   - Filters remaining invalid records.
   - Drops the 'status' column for model input cleanliness.

------------------------------------------------------------------------------------
Expected Output
------------------------------------------------------------------------------------
Returns a clean, consistent, memory-optimized `pandas.DataFrame` with:
   - Verified and enriched transactions
   - Minimal noise and redundancy
   - Sybil address labels (`is_sybil`)
   - Inclusion timestamps (`inclusion_time`)
   - Miner flag (`is_miner`)

------------------------------------------------------------------------------------
Typical Use Case
------------------------------------------------------------------------------------
This pipeline is primarily used in:
   - Blockchain forensic research (MEV, front-running, Sybil behavior)
   - Preprocessing before feature engineering for machine learning
   - Exploratory data analysis in DeFi security audits

------------------------------------------------------------------------------------
Dependencies
------------------------------------------------------------------------------------
- pandas >= 1.0
- numpy >= 1.18
- logging (standard library)

"""


In [ ]:
import pandas as pd
import numpy as np

CRITICAL_COLUMNS = ['detecttime', 'hash', 'curblocknumber',
                    'fromaddress', 'toaddress', 'status']

IRRELEVANT_COLUMNS = ['blobversionedhashes', 'maxfeeperblobgas', 'network', 'stuck', 'reorg',
                      'replace', 'failurereason', 'dropreason', 'rejectionreason', 'drop_reason',
                      'detect_date', 'was_evicted', 'was_rejected', 'rejection_reason',
                      'region', 'time_pending']


def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    bad_cols = [col for col in df.columns if ',' in col or '\n' in col or col.strip() == '']
    df = df.drop(columns=bad_cols, errors='ignore')
    df = df.rename(columns={col: col.strip().replace('\n', '') for col in df.columns})
    df = df.loc[:, ~df.columns.duplicated()]
    return df


def drop_irrelevant_columns(df: pd.DataFrame, irrelevant_cols: list) -> pd.DataFrame:
    df = df.drop(columns=[col for col in irrelevant_cols if col in df.columns], errors='ignore')
    return df


def drop_missing_critical(df: pd.DataFrame, critical_cols: list) -> pd.DataFrame:
    available_cols = [col for col in critical_cols if col in df.columns]
    df = df.dropna(subset=available_cols)
    return df


def final_confirmed_dedup(df: pd.DataFrame) -> pd.DataFrame:
    if 'status' in df.columns:
        df = df[df['status'] == 'confirmed']
    if 'timepending' in df.columns:
        df = df[df['timepending'].notna()]
    if 'blockspending' in df.columns:
        df = df[df['blockspending'].notna()]
    if 'region' in df.columns:
        df = df.drop(columns=['region'])
    df = df.drop_duplicates()
    return df


def label_sybil_addresses(df: pd.DataFrame, is_sybil_df: pd.DataFrame) -> pd.DataFrame:
    df['fromaddress'] = df['fromaddress'].str.lower()
    is_sybil_df['address'] = is_sybil_df['address'].str.lower()
    df = df.merge(is_sybil_df[['address', 'is_sybil']], how='left',
                  left_on='fromaddress', right_on='address')
    df = df.drop(columns=['address'], errors='ignore')
    df['is_sybil'] = df['is_sybil'].fillna(0).astype(int)
    return df


def compute_inclusion_time_and_miner_flag(df: pd.DataFrame) -> pd.DataFrame:
    if 'detecttime' in df.columns and 'timepending' in df.columns:
        df['detecttime'] = pd.to_datetime(df['detecttime'], utc=True)
        df['timepending'] = pd.to_numeric(df['timepending'], errors='coerce')
        df['inclusion_time'] = df['detecttime'] + pd.to_timedelta(df['timepending'] / 1000, unit='s')
        df['is_miner'] = df['timepending'].apply(lambda x: 1 if x == 0 else 0)
    return df


def filter_confirmed_only(df: pd.DataFrame) -> pd.DataFrame:
    if 'status' in df.columns and 'timepending' in df.columns:
        df = df[(df['status'] == 'confirmed') & (df['timepending'].notna())]
    return df


def drop_column_if_exists(df: pd.DataFrame, column_name: str) -> pd.DataFrame:
    if column_name in df.columns:
        df = df.drop(columns=[column_name])
    return df


def full_modular_pipeline_optimized(df: pd.DataFrame,
                                    irrelevant_cols: list,
                                    critical_cols: list,
                                    is_sybil_df: pd.DataFrame) -> pd.DataFrame:
    df = clean_column_names(df)
    df = drop_irrelevant_columns(df, irrelevant_cols)
    df = drop_missing_critical(df, critical_cols)
    df = final_confirmed_dedup(df)
    df = label_sybil_addresses(df, is_sybil_df)
    df = compute_inclusion_time_and_miner_flag(df)
    df = filter_confirmed_only(df)
    df = drop_column_if_exists(df, "status")
    return df
